In [85]:
import pandas as pd
import numpy as np

In [86]:
PATH = './data'
TARGET = 'SalePrice'

In [87]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

In [88]:
df = pd.read_csv(PATH + "/train.csv")

In [89]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, shuffle=True, random_state=1337
)

train_ids = X_train["Id"]
test_ids = X_test["Id"]

print(f"X_train shape {X_train.shape}")
print(f"X_test shape {X_test.shape}")

X_train shape (1168, 80)
X_test shape (292, 80)


### Data Cleaning ###

### Categorical Imputing ###

In [90]:
# we first impute all missing values in categorical and numeric features.

# These categorical columns may have 'NaN' values in them which means missing.
# data_description.txt did not mention 'Nan' was intended.

cat_impute_cols = [
    'MSZoning',
    'Street',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'Foundation',
    'ExterQual',
    'ExterCond',
    'Heating',
    'HeatingQC',
    'CentralAir',
    'Electrical',
    'KitchenQual',
    'Functional',
    'PavedDrive',
    'SaleType',
    'SaleCondition',
]

### Numeric Imputing ###

In [91]:
# We impute all numeric columns because data_description.txt did not mention any numeric feature with intended 'NaN' value.

# num_impute_cols = [
#     'LotFrontage',
#     'GarageYrBlt',
#     'MasVnrArea',
# ]

num_impute_cols = list(X_train.select_dtypes(exclude='object').columns)
num_impute_cols.remove('Id')
num_impute_cols.remove('MSSubClass')

### Categorical One-Hot-Encoded Columns ###

In [92]:
# Here we list features to be one-hot-encoded including 'NaN's because 'NaN' here does not mean missing.
# 'MasVnrType' was interesting column here.

cat_ohe_cols = [
    'MSZoning',
    'Street',
    'Alley',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'MasVnrType',
    'Foundation',
    'Heating',
    'CentralAir',
    'Electrical',
    'GarageType',
    'PavedDrive',
    'MiscFeature',
    'SaleType',
    'SaleCondition',
]

cat_ohe_missing_cols = [
    'MSZoning',
    'Street',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'Foundation',
    'Heating',
    'CentralAir',
    'Electrical',
    'PavedDrive',
    'SaleType',
    'SaleCondition'
]

cat_ohe_absent_cols = [
    'Alley',
    'MasVnrType',
    'GarageType',
    'MiscFeature'
]

### Ordinal Columns ###

In [93]:
# Some categorical columns can may be ordered, replace them with numbers.

cat_ord_cols = [
    'ExterQual',
    'ExterCond',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'HeatingQC',
    'KitchenQual',
    'Functional',
    'FireplaceQu',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'Fence',
]

cat_ord_missing_cols = [
    'ExterQual',
    'ExterCond',
    'HeatingQC',
    'KitchenQual',
    'Functional',
]

cat_ord_absent_cols = [
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'FireplaceQu',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'Fence',
]

exter_qual_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
exter_cond_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_qual_order         = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_cond_order         = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_exposure_order     = ['NA', 'No', 'Mn', 'Av', 'Gd']
bsmt_fin_type_1_order   = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
bsmt_fin_type_2_order   = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
heating_qc_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
kitchen_qual_order      = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
functional_order        = ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']
fireplace_qu_order      = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
garage_finish_order     = ['NA', 'Unf', 'RFn', 'Fin']
garage_qual_order       = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
garage_cond_order       = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
pool_qc_order           = ['NA', 'Fa', 'TA', 'Gd', 'Ex']
fence_order             = ['NA', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']

cat_ord_missing_categories = [
    exter_qual_order,
    exter_cond_order,
    heating_qc_order,
    kitchen_qual_order,
    functional_order,
]

cat_ord_absent_categories = [
    bsmt_qual_order,
    bsmt_cond_order,
    bsmt_exposure_order,
    bsmt_fin_type_1_order,
    bsmt_fin_type_2_order,
    fireplace_qu_order,
    garage_finish_order,
    garage_qual_order,
    garage_cond_order,
    pool_qc_order,
    fence_order,
]

### Numeric One-Hot-Encoded Columns ###

In [94]:
# This feature has numeric type which is misleading.

num_ohe_cols = [
    'MSSubClass',
]

### Data Cleanup Pipeline ###

In [99]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

num_impute_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='median')),
    ('scale',   StandardScaler())
])

num_ohe_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

cat_ord_missing_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=cat_ord_missing_categories)),
])

cat_ord_absent_pipeline = Pipeline([
    ('nan_impute',  SimpleImputer(strategy='constant', missing_values=np.nan, fill_value='NA')),
    ('ordinal',     OrdinalEncoder(categories=cat_ord_absent_categories)),
])

cat_ohe_missing_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

cat_ohe_absent_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='constant', missing_values=np.nan, fill_value='Missing')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num_impute',       num_impute_pipeline,       num_impute_cols),
        ('num_ohe',          num_ohe_pipeline,          num_ohe_cols),
        ('cat_ord_absent',   cat_ord_absent_pipeline,   cat_ord_absent_cols),
        ('cat_ord_missing',  cat_ord_missing_pipeline,  cat_ord_missing_cols),
        ('cat_ohe_absent',   cat_ohe_absent_pipeline,   cat_ohe_absent_cols),
        ('cat_ohe_missing',  cat_ohe_missing_pipeline,  cat_ohe_missing_cols),
        ('drop',    'drop',                             ['Id']),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
)

preprocessor.set_output(transform='pandas')

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_impute', ...), ('num_ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and `

In [100]:
# scaling every column after preprocessing decreased accuracy.
# scaling only numeric columns is sufficient.

preprocessor_pipeline = Pipeline([
    ('preprocessor', preprocessor),
])

preprocessor_pipeline.set_output(transform='pandas')

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_impute', ...), ('num_ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

In [101]:
y_train_t = np.log1p(y_train)
y_test_t = np.log1p(y_test)

preprocessor_pipeline.fit(X_train, y_train_t)

X_train_t = preprocessor_pipeline.transform(X_train)
X_test_t = preprocessor_pipeline.transform(X_test)

### Full Pipeline (Linear Regression) ###

In [104]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', LinearRegression())
])

param_grid = {
    'preprocessor__preprocessor__num_impute__impute__strategy': ['mean', 'median'],
}

search = GridSearchCV(
    final_pipeline,
    param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
)

search.fit(X_train, y_train_t)
cv_results_df = pd.DataFrame(search.cv_results_)

cv_results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_preprocessor__preprocessor__num_impute__impute__strategy,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,0.360777,0.259073,0.083052,0.026285,mean,{'preprocessor__preprocessor__num_impute__impu...,-0.163426,-0.142485,-0.203979,-0.172463,-0.182621,-0.172995,0.020373,2,-0.088076,-0.089692,-0.091193,-0.091805,-0.085969,-0.089347,0.002126
1,0.172910,0.045459,0.071261,0.005420,median,{'preprocessor__preprocessor__num_impute__impu...,-0.163390,-0.142456,-0.203974,-0.172457,-0.182637,-0.172983,0.020385,1,-0.088084,-0.089698,-0.091192,-0.091808,-0.085969,-0.089350,0.002126


### Full Pipeline (Ridge) ###

In [110]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Ridge())
])

param_grid = {
    'preprocessor__preprocessor__num_impute__impute__strategy': ['mean', 'median'],
    # 'model__alpha': [.001, .01, .05, .1, .5, 1, 5, 9, 9.1, 9.2, 9.3, 9.4],
    # 'model__alpha': np.linspace(.01, 50, 300),
    'model__alpha': np.logspace(np.log10(9.5), np.log10(10), 300),
}

search = GridSearchCV(
    final_pipeline,
    param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
    verbose=1,
    n_jobs=-1
)

search.fit(X_train, y_train_t)
cv_results_df = pd.DataFrame(search.cv_results_)

cv_results_df

Fitting 5 folds for each of 600 candidates, totalling 3000 fits


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__alpha,param_preprocessor__preprocessor__num_impute__impute__strategy,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,0.203488,0.038527,0.122485,0.027955,9.500000,mean,"{'model__alpha': 9.499999999999998, 'preproces...",-0.134634,-0.134157,-0.202748,-0.115203,-0.139021,-0.145153,0.029945,300,-0.111555,-0.111831,-0.101545,-0.114940,-0.111117,-0.110198,0.004532
1,0.217563,0.035363,0.125340,0.036894,9.500000,median,"{'model__alpha': 9.499999999999998, 'preproces...",-0.134636,-0.134160,-0.202736,-0.115206,-0.139057,-0.145159,0.029937,600,-0.111558,-0.111825,-0.101547,-0.114934,-0.111112,-0.110195,0.004530
2,0.315310,0.090939,0.141205,0.026769,9.501630,mean,"{'model__alpha': 9.5016298598526, 'preprocesso...",-0.134634,-0.134157,-0.202749,-0.115203,-0.139020,-0.145153,0.029945,299,-0.111556,-0.111832,-0.101546,-0.114941,-0.111119,-0.110199,0.004532
3,0.395334,0.111604,0.285780,0.109181,9.501630,median,"{'model__alpha': 9.5016298598526, 'preprocesso...",-0.134636,-0.134161,-0.202736,-0.115206,-0.139057,-0.145159,0.029937,599,-0.111559,-0.111826,-0.101548,-0.114935,-0.111113,-0.110196,0.004530
4,0.457636,0.154157,0.164535,0.046811,9.503260,mean,"{'model__alpha': 9.503259999330792, 'preproces...",-0.134634,-0.134158,-0.202749,-0.115203,-0.139020,-0.145153,0.029945,298,-0.111557,-0.111833,-0.101547,-0.114942,-0.111120,-0.110200,0.004532
5,0.400911,0.155299,0.185379,0.079468,9.503260,median,"{'model__alpha': 9.503259999330792, 'preproces...",-0.134636,-0.134161,-0.202736,-0.115206,-0.139056,-0.145159,0.029938,598,-0.111560,-0.111827,-0.101549,-0.114936,-0.111114,-0.110197,0.004530
6,0.366657,0.121878,0.186484,0.061729,9.504890,mean,"{'model__alpha': 9.504890418482551, 'preproces...",-0.134634,-0.134158,-0.202749,-0.115203,-0.139020,-0.145153,0.029945,297,-0.111558,-0.111834,-0.101548,-0.114944,-0.111121,-0.110201,0.004533
7,0.412411,0.131642,0.141532,0.032443,9.504890,median,"{'model__alpha': 9.504890418482551, 'preproces...",-0.134636,-0.134161,-0.202737,-0.115206,-0.139056,-0.145159,0.029938,597,-0.111561,-0.111828,-0.101549,-0.114937,-0.111115,-0.110198,0.004530
8,0.267483,0.056481,0.147536,0.061540,9.506521,mean,"{'model__alpha': 9.506521117355861, 'preproces...",-0.134634,-0.134158,-0.202750,-0.115203,-0.139020,-0.145153,0.029945,296,-0.111559,-0.111835,-0.101549,-0.114945,-0.111122,-0.110202,0.004533
9,0.350318,0.097767,0.137119,0.031487,9.506521,median,"{'model__alpha': 9.506521117355861, 'preproces...",-0.134636,-0.134161,-0.202737,-0.115206,-0.139056,-0.145159,0.029938,596,-0.111562,-0.111829,-0.101550,-0.114938,-0.111116,-0.110199,0.004530


### Full Pipeline (Lasso) ###

In [115]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Lasso(max_iter=10000))
])

param_grid = {
    'preprocessor__preprocessor__num_impute__impute__strategy': ['mean', 'median'],
    # 'model__alpha': [.0005, .0006, .0007, .0008, .0009, .001, .005, .01, .05, .1, .5, 1, 5, 10],
    'model__alpha': np.logspace(-5, 1, 300),
}

search = GridSearchCV(
    final_pipeline,
    param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
    verbose=1,
    n_jobs=-1
)

search.fit(X_train, y_train_t)
cv_results_df = pd.DataFrame(search.cv_results_)

cv_results_df

Fitting 5 folds for each of 600 candidates, totalling 3000 fits


KeyboardInterrupt: 

### Final Pipeline (ElasticNet) ###

In [117]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', ElasticNet(max_iter=10000))
])

param_grid = {
    'preprocessor__preprocessor__num_impute__impute__strategy':
        ['mean', 'median'],
    # 'model__alpha':
    #     [.0005, .001, .002, .005, .1, .5, 1],
    # 'model__l1_ratio':
    #     [.1, .2, .3, .4, .5, .6, .7, .8, .9],

    'model__alpha':
        np.logspace(-4, 1, 30),
    'model__l1_ratio':
        np.linspace(.05, .95, 20)
}

# {\'model__alpha\': 0.0007278953843983154, \'model__l1_ratio\': 0.7131578947368421, \'preprocessor__preprocessor__num_impute__impute__strategy\': \'mean\'}

search = GridSearchCV(
    final_pipeline,
    param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
    verbose=1,
    n_jobs=-1,
)

search.fit(X_train, y_train_t)
cv_results_df = pd.DataFrame(search.cv_results_)
cv_results_df

Fitting 5 folds for each of 1200 candidates, totalling 6000 fits


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__alpha,param_model__l1_ratio,param_preprocessor__preprocessor__num_impute__impute__strategy,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,1.849657,0.451527,0.175588,0.052266,0.000100,0.050000,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.154017,-0.139171,-0.202787,-0.151965,-0.166145,-0.162817,0.021739,634,-0.089031,-0.090365,-0.091417,-0.092494,-0.086947,-0.090051,0.001929
1,2.169421,0.375109,0.270007,0.097461,0.000100,0.050000,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.153982,-0.139145,-0.202781,-0.151953,-0.166152,-0.162803,0.021747,633,-0.089039,-0.090371,-0.091417,-0.092497,-0.086948,-0.090054,0.001929
2,1.911086,0.316967,0.196060,0.090735,0.000100,0.097368,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.153676,-0.138782,-0.201984,-0.150294,-0.164840,-0.161915,0.021689,626,-0.089098,-0.090423,-0.091491,-0.092545,-0.087000,-0.090111,0.001931
3,1.892845,0.449509,0.150945,0.027947,0.000100,0.097368,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.153639,-0.138755,-0.201977,-0.150283,-0.164848,-0.161900,0.021697,625,-0.089106,-0.090428,-0.091491,-0.092548,-0.087000,-0.090115,0.001930
4,1.607182,0.620126,0.172680,0.041286,0.000100,0.144737,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.153315,-0.138304,-0.201901,-0.148549,-0.163493,-0.161113,0.021946,614,-0.089176,-0.090491,-0.091579,-0.092604,-0.087060,-0.090182,0.001932
5,1.665111,0.433308,0.178837,0.065865,0.000100,0.144737,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.153278,-0.138279,-0.201896,-0.148536,-0.163500,-0.161098,0.021954,613,-0.089184,-0.090496,-0.091579,-0.092606,-0.087061,-0.090185,0.001932
6,1.102434,0.269880,0.133943,0.016410,0.000100,0.192105,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.152980,-0.137553,-0.201986,-0.146697,-0.162269,-0.160297,0.022350,602,-0.089261,-0.090558,-0.091675,-0.092670,-0.087123,-0.090258,0.001936
7,1.246121,0.200673,0.148065,0.028619,0.000100,0.192105,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.152935,-0.137529,-0.201979,-0.146699,-0.162275,-0.160284,0.022355,601,-0.089269,-0.090564,-0.091675,-0.092673,-0.087124,-0.090261,0.001936
8,1.137203,0.347783,0.140208,0.016574,0.000100,0.239474,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.152648,-0.136979,-0.202184,-0.144841,-0.161349,-0.159600,0.022778,588,-0.089352,-0.090635,-0.091776,-0.092728,-0.087171,-0.090332,0.001943
9,0.961901,0.221209,0.139955,0.016369,0.000100,0.239474,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.152599,-0.136960,-0.202179,-0.144846,-0.161354,-0.159588,0.022783,586,-0.089360,-0.090640,-0.091776,-0.092731,-0.087172,-0.090336,0.001942


In [121]:
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', ElasticNet(alpha=0.0007278953843983154, l1_ratio=0.7131578947368421))
])
final_pipeline.fit(X_train, y_train_t)
y_pred = final_pipeline.predict(X_test)
print(root_mean_squared_error(y_test_t, y_pred))

y_pred = final_pipeline.predict(X_train)
print(root_mean_squared_error(y_train_t, y_pred))



# final_pipeline = Pipeline([
#     ('preprocessor', preprocessor_pipeline),
#     ('model', Lasso(alpha=.0007934096665797492))
# ])
# final_pipeline.fit(X_train, y_train_t)
# y_pred = final_pipeline.predict(X_test)
# print(root_mean_squared_error(y_test_t, y_pred))
#
# y_pred = final_pipeline.predict(X_train)
# print(root_mean_squared_error(y_train_t, y_pred))
#
#
# final_pipeline = Pipeline([
#     ('preprocessor', preprocessor_pipeline),
#     ('model', LinearRegression())
# ])
# final_pipeline.fit(X_train, y_train_t)
# y_pred = final_pipeline.predict(X_test)
# print(root_mean_squared_error(y_test_t, y_pred))
#
# y_pred = final_pipeline.predict(X_train)
# print(root_mean_squared_error(y_train_t, y_pred))

0.11544915376776345
0.10881045553882318
0.1156369745205326
0.11243910374712052
0.12966410509994647
0.09419088615888568


In [189]:
final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Ridge(alpha=400))
])

cv_results = cross_validate(
    final_pipeline, X_train, y_train_t,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
)

cv_results_df = pd.DataFrame(cv_results)

train_root_mean_squared_error = -cv_results_df.mean()['train_score']
validation_root_mean_squared_error = -cv_results_df.mean()['test_score']

print(f'train_rmse: {train_root_mean_squared_error}')
print(f'validation_rmse: {validation_root_mean_squared_error}')

train_rmse: 0.10310912196089887
validation_rmse: 0.13957445547139208


In [118]:
final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', ElasticNet(alpha=0.0007278953843983154, l1_ratio=0.7131578947368421))
])
test = pd.read_csv(PATH + '/test.csv')
test_ids = test['Id']

# final_pipeline.fit(pd.concat([X_train, X_test], axis=0, ignore_index=True), pd.concat([y_train_t, y_test_t], axis=0, ignore_index=True))
final_pipeline.fit(X_train, y_train_t)
prices_log = final_pipeline.predict(test)
prices = np.expm1(prices_log)

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": prices,
})

submission.to_csv("submission.csv", index=False)